# FloodShift — Proposed Reliability-Aware Multimodal Segmentation Model

**Notebook:** `04B_FloodShift_Proposed_Model_READY`

This notebook implements the proposed FloodShift architecture discussed during project development:

- separate Sentinel-1 SAR and Sentinel-2 optical encoders
- SE channel attention
- multi-scale spatial reliability routing
- frozen OmniCloudMask `P(clear)` as the optical-reliability prior
- Fusion Transformer at the bottleneck
- attention-gated skip connections
- 3-class segmentation
- differential learning rates
- calibrated uncertainty / abstention
- counterfactual SAR-vs-optical sensor attribution

## Kaggle inputs required

Attach:

1. `01_FloodShift_Data_Audit`
2. the final **successful saved output** of `03_FloodShift_Preprocessing_FINAL_COMPLETE`

`04A_FloodShift_Standard_Baselines_8Models_READY` is **not required to train this model**. Attach its successful output only later if you want the optional comparison cell.

## Main comparison rule

The main proposed-model run uses **Raw SAR** so its architecture can be compared fairly against the 8-model baseline notebook, which also uses Raw SAR. The frozen Lee Sigma / BM3D / DP-SSD branches remain a separate preprocessing ablation.

## Architecture

```text
SAR VV/VH ──► SAR Encoder ───────────┐
                                     │
P(clear) ─► multi-scale Reliability Routers ─► weighted fused features
                                     │
10-band S2 ─► Optical Encoder ───────┘
                                     │
                            Fusion Transformer
                              (16×16 = 256 tokens)
                                     │
                         attention-gated decoder
                                     │
                             3-class logits
```

At each scale:

\[
w_{SAR}(x,y)+w_{OPT}(x,y)=1
\]

and

\[
F=w_{SAR}F_{SAR}+w_{OPT}F_{OPT}.
\]

`P(clear)` is a frozen preprocessing prior, not a flood-label-trained cloud mask.

In [2]:
# ============================================================
# CELL: P4-C01 — ENVIRONMENT + REPRODUCIBILITY
# SINGLE-GPU SAFE: avoid nn.DataParallel CUDA issues
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # before importing torch

import sys
import gc
import json
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

torch.backends.cudnn.benchmark = True
AMP_ENABLED = torch.cuda.is_available()

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_Proposed")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration"
ATTRIBUTION_ROOT = OUTPUT_ROOT / "attribution"

for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, METRIC_ROOT, CALIBRATION_ROOT, ATTRIBUTION_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 108)
print("P4-C01 — FLOODSHIFT PROPOSED MODEL [SINGLE-GPU SAFE]")
print("=" * 108)
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)

assert torch.cuda.device_count() <= 1


P4-C01 — FLOODSHIFT PROPOSED MODEL [SINGLE-GPU SAFE]
Python      : 3.12.13
PyTorch     : 2.10.0+cu128
Device      : cuda:0
Visible GPUs: 1
GPU         : Tesla T4
AMP         : True
Output root : /kaggle/working/FloodShift_Proposed


In [5]:
# ============================================================
# CELL: P4-C02 — LOCATE FROZEN NOTEBOOK-03 ARTIFACTS
# ============================================================

_file_cache = {}

def locate_by_saved_path(saved_path_or_name):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)

    if cache_key in _file_cache:
        return _file_cache[cache_key]

    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not locate '{filename}'. "
            "Attach Notebook 01 and the final Notebook 03 output."
        )

    parent_hint = saved.parent.name.lower()
    hinted = [
        p for p in matches
        if parent_hint and parent_hint in str(p).lower()
    ]

    candidates = hinted if hinted else matches

    selected = sorted(
        candidates,
        key=lambda p: (len(str(p)), str(p))
    )[0]

    _file_cache[cache_key] = selected
    return selected


FINAL_MANIFEST_PATH = locate_by_saved_path(
    "floodshift_preprocessing_manifest_v1.csv"
)

GLOBAL_LOCK_PATH = locate_by_saved_path(
    "floodshift_preprocessing_GLOBAL_LOCK_v1.json"
)

OPTICAL_NORM_PATH = locate_by_saved_path(
    "optical_10band_train_normalization_exact.csv"
)

SAR_SCENE_STATS_PATH = locate_by_saved_path(
    "train_raw_sar_scene_statistics.csv"
)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 104)
print("P4-C02 — FROZEN PREPROCESSING FOUND")
print("=" * 104)

for p in [
    FINAL_MANIFEST_PATH,
    GLOBAL_LOCK_PATH,
    OPTICAL_NORM_PATH,
    SAR_SCENE_STATS_PATH
]:
    print(" ", p)

print("\nLock:", preprocessing_lock["status"])


P4-C02 — FROZEN PREPROCESSING FOUND
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_manifest_v1.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/final_preprocessing/floodshift_preprocessing_GLOBAL_LOCK_v1.json
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/optical/optical_10band_train_normalization_exact.csv
  /kaggle/input/notebooks/himavarshini2108/03-floodshift-preprocessing-final-complete-kaggle/FloodShift_Preprocessing/train_raw_sar_scene_statistics.csv

Lock: PERMANENTLY_FROZEN


In [6]:
# ============================================================
# CELL: P4-C03 — MANIFEST + TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = manifest[manifest["split"] == "train"].copy().reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].copy().reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].copy().reset_index(drop=True)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = np.isfinite(mu_all) & np.isfinite(sd_all) & (n_all > 0)

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std("vv_mean_db", "vv_std_db")
VH_MEAN, VH_STD = pooled_mean_std("vh_mean_db", "vh_std_db")

SAR_MEAN = np.array([VV_MEAN, VH_MEAN], dtype=np.float32)
SAR_STD = np.array([VV_STD, VH_STD], dtype=np.float32)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 104)
print("P4-C03 — DATA CONFIG")
print("=" * 104)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : renormalized P(clear)")
print("Main SAR branch    : Raw SAR")


P4-C03 — DATA CONFIG
Train / Val / Test: 314 68 58
Optical bands      : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
SAR mean [VV,VH]   : [-10.358468 -17.10712 ]
SAR std  [VV,VH]   : [4.161632  4.8134336]
Reliability        : renormalized P(clear)
Main SAR branch    : Raw SAR


In [7]:
# ============================================================
# CELL: P4-C04 — TARGET + DATASET
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3
CLASS_NAMES = ["non_flood", "permanent_water", "temporary_flood"]

TILE_SIZE = 256
TILES_PER_SCENE = 4
TILE_OFFSETS = [(0,0), (0,256), (256,0), (256,256)]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(row["optical_reliability_path"]),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(flood.shape, IGNORE_INDEX, dtype=np.uint8)

    valid = (flood != -1)

    target[valid] = 0
    target[valid & (jrc == 1)] = 1
    target[valid & (jrc != 1) & (flood == 1)] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(np.float32, copy=False)

    valid = np.isfinite(sar).all(axis=0)
    out = np.zeros_like(sar, dtype=np.float32)

    for c in range(2):
        m = valid & np.isfinite(sar[c])
        out[c, m] = (sar[c, m] - SAR_MEAN[c]) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(np.float32, copy=False)

    valid = np.all(
        np.isfinite(optical_13) & (optical_13 != 0),
        axis=0
    )

    reflectance = optical_13 / 10000.0
    optical_10 = reflectance[OPTICAL_SOURCE_INDICES]

    out = (
        optical_10 - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(np.float32, copy=False)
    out[:, ~valid] = 0.0

    return out


def augment_all(sar, optical, reliability, target):
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(reliability, axis=1)
        target = np.flip(target, axis=1)

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(reliability, axis=0)
        target = np.flip(target, axis=0)

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(sar, k=k, axes=(1,2))
        optical = np.rot90(optical, k=k, axes=(1,2))
        reliability = np.rot90(reliability, k=k, axes=(0,1))
        target = np.rot90(target, k=k, axes=(0,1))

    return sar.copy(), optical.copy(), reliability.copy(), target.copy()


class FloodShiftDataset(Dataset):
    def __init__(self, dataframe, train_mode=False):
        self.df = dataframe.reset_index(drop=True)
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df) * TILES_PER_SCENE

    def __getitem__(self, index):
        scene_index = index // TILES_PER_SCENE
        tile_index = index % TILES_PER_SCENE

        row = self.df.iloc[scene_index]
        scene_id = str(row["scene_id"])

        y0, x0 = TILE_OFFSETS[tile_index]
        window = Window(x0, y0, TILE_SIZE, TILE_SIZE)

        paths = resolve_scene_files(row)

        with rasterio.open(paths["sar"]) as src:
            sar = src.read(window=window).astype(np.float32)

        with rasterio.open(paths["optical"]) as src:
            optical_13 = src.read(window=window).astype(np.float32)

        with rasterio.open(paths["flood"]) as src:
            flood = src.read(1, window=window)

        with rasterio.open(paths["jrc"]) as src:
            jrc = src.read(1, window=window)

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r"
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0+TILE_SIZE,
                x0:x0+TILE_SIZE
            ],
            dtype=np.float32
        )

        sar = normalize_sar(sar)
        optical = normalize_optical(optical_13)
        target = build_three_class_target(flood, jrc)

        reliability = np.clip(reliability, 0.0, 1.0)

        if self.train_mode:
            sar, optical, reliability, target = augment_all(
                sar, optical, reliability, target
            )

        return {
            "sar": torch.from_numpy(sar),
            "optical": torch.from_numpy(optical),
            "reliability": torch.from_numpy(reliability[None, ...]),
            "target": torch.from_numpy(target.astype(np.int64)),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }


In [8]:
# ============================================================
# CELL: P4-C05 — TARGET AUDIT + CLASS WEIGHTS
# ============================================================

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
ignored_count = 0

for position, (_, row) in enumerate(train_df.iterrows(), start=1):
    paths = resolve_scene_files(row)

    with rasterio.open(paths["flood"]) as src:
        flood = src.read(1)

    with rasterio.open(paths["jrc"]) as src:
        jrc = src.read(1)

    target = build_three_class_target(flood, jrc)

    for c in range(NUM_CLASSES):
        class_counts[c] += int((target == c).sum())

    ignored_count += int((target == IGNORE_INDEX).sum())

    if position == 1 or position % 75 == 0 or position == len(train_df):
        print(f"[{position:03d}/{len(train_df)}] target audit")

valid_total = class_counts.sum()
frequencies = class_counts / valid_total

weights_np = 1.0 / np.maximum(frequencies, 1e-12)
weights_np = weights_np / weights_np.mean()

CLASS_WEIGHTS = torch.tensor(
    weights_np,
    dtype=torch.float32,
    device=DEVICE
)

target_audit = pd.DataFrame({
    "class_id": np.arange(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "pixel_count": class_counts,
    "frequency": frequencies,
    "loss_weight": weights_np,
})

print("\n" + "=" * 104)
print("P4-C05 — TRAIN TARGET DISTRIBUTION")
print("=" * 104)
print(target_audit.to_string(index=False))
print("\nIgnored:", f"{ignored_count:,}")
print("Loss weights:", CLASS_WEIGHTS.detach().cpu().numpy())

target_audit.to_csv(
    METRIC_ROOT / "proposed_train_target_distribution.csv",
    index=False
)

assert (class_counts > 0).all()


[001/314] target audit
[075/314] target audit
[150/314] target audit
[225/314] target audit
[300/314] target audit
[314/314] target audit

P4-C05 — TRAIN TARGET DISTRIBUTION
 class_id      class_name  pixel_count  frequency  loss_weight
        0       non_flood     64724902   0.892878     0.063367
        1 permanent_water      1826059   0.025190     2.246068
        2 temporary_flood      5939270   0.081932     0.690565

Ignored: 9,822,985
Loss weights: [0.06336745 2.2460675  0.690565  ]


In [9]:
# ============================================================
# CELL: P4-C06 — DATALOADERS + ONE-BATCH CHECK
# ============================================================

BATCH_SIZE = 8
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_dataset = FloodShiftDataset(train_df, train_mode=True)
val_dataset = FloodShiftDataset(val_df, train_mode=False)
test_dataset = FloodShiftDataset(test_df, train_mode=False)

def make_loaders(seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)

    common = {
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "persistent_workers": (NUM_WORKERS > 0),
    }

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=g,
        **common
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        **common
    )

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()

batch = next(iter(train_loader))

print("=" * 104)
print("P4-C06 — MULTIMODAL BATCH CHECK")
print("=" * 104)

print("SAR        :", tuple(batch["sar"].shape))
print("Optical    :", tuple(batch["optical"].shape))
print("Reliability:", tuple(batch["reliability"].shape))
print("Target     :", tuple(batch["target"].shape))
print(
    "P(clear)   :",
    float(batch["reliability"].min()),
    "to",
    float(batch["reliability"].max())
)

assert batch["sar"].shape[1:] == (2,256,256)
assert batch["optical"].shape[1:] == (10,256,256)
assert batch["reliability"].shape[1:] == (1,256,256)
assert batch["target"].shape[1:] == (256,256)

assert torch.isfinite(batch["sar"]).all()
assert torch.isfinite(batch["optical"]).all()
assert torch.isfinite(batch["reliability"]).all()


P4-C06 — MULTIMODAL BATCH CHECK
SAR        : (8, 2, 256, 256)
Optical    : (8, 10, 256, 256)
Reliability: (8, 1, 256, 256)
Target     : (8, 256, 256)
P(clear)   : 0.000995635986328125 to 0.9970703125


## Network components

The proposed network keeps the discussed components explicit:

1. **SE channel attention** inside each modality encoder.
2. **ReliabilityRouter** at four scales.
3. **FusionTransformer** at the 16×16 bottleneck. For a 256×256 tile this gives **256 spatial tokens**.
4. **Attention-gated decoder skips**.

The modality encoders are implemented separately from the first layer: SAR starts with 2 channels and optical starts with 10 channels.

In [10]:
# ============================================================
# CELL: P4-C07 — FLOODSHIFT ARCHITECTURE MODULES
# ============================================================

def make_group_norm(channels):
    for groups in [32, 16, 8, 4, 2, 1]:
        if channels % groups == 0:
            return nn.GroupNorm(groups, channels)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        hidden = max(channels // reduction, 8)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.GELU(),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))


class BasicBlockSE(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )

        self.norm1 = make_group_norm(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, padding=1, bias=False
        )

        self.norm2 = make_group_norm(out_channels)
        self.se = SEBlock(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False
                ),
                make_group_norm(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

        self.act = nn.GELU()

    def forward(self, x):
        residual = self.shortcut(x)

        x = self.act(
            self.norm1(
                self.conv1(x)
            )
        )

        x = self.norm2(
            self.conv2(x)
        )

        x = self.se(x)
        x = x + residual

        return self.act(x)


class ModalityEncoder(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                in_channels, 64,
                kernel_size=5, stride=2, padding=2, bias=False
            ),
            make_group_norm(64),
            nn.GELU()
        )

        self.stage1 = self._make_stage(64, 64, blocks=2, stride=1)
        self.stage2 = self._make_stage(64, 128, blocks=2, stride=2)
        self.stage3 = self._make_stage(128, 256, blocks=2, stride=2)
        self.stage4 = self._make_stage(256, 512, blocks=2, stride=2)

    @staticmethod
    def _make_stage(in_channels, out_channels, blocks, stride):
        layers = [
            BasicBlockSE(
                in_channels, out_channels, stride=stride
            )
        ]

        for _ in range(blocks - 1):
            layers.append(
                BasicBlockSE(
                    out_channels, out_channels, stride=1
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)

        s1 = self.stage1(x)  # 64 x 128 x 128
        s2 = self.stage2(s1) # 128 x 64 x 64
        s3 = self.stage3(s2) # 256 x 32 x 32
        s4 = self.stage4(s3) # 512 x 16 x 16

        return [s1, s2, s3, s4]


class ReliabilityRouter(nn.Module):
    def __init__(self, channels):
        super().__init__()

        hidden = min(
            max(channels // 4, 16),
            64
        )

        self.predictor = nn.Sequential(
            nn.Conv2d(
                2 * channels + 1,
                hidden,
                kernel_size=1,
                bias=False
            ),
            make_group_norm(hidden),
            nn.GELU(),

            nn.Conv2d(
                hidden,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            make_group_norm(hidden),
            nn.GELU(),

            nn.Conv2d(
                hidden,
                2,
                kernel_size=1
            )
        )

    def forward(self, sar_feature, optical_feature, p_clear):
        reliability = F.interpolate(
            p_clear,
            size=sar_feature.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        router_input = torch.cat(
            [sar_feature, optical_feature, reliability],
            dim=1
        )

        logits = self.predictor(router_input)

        # Reliability prior is applied only to the optical routing logit.
        optical_prior = torch.clamp(
            reliability,
            min=1e-3,
            max=1.0
        )

        logits = torch.cat(
            [
                logits[:, 0:1],
                logits[:, 1:2] + torch.log(optical_prior)
            ],
            dim=1
        )

        weights = torch.softmax(
            logits,
            dim=1
        )

        fused = (
            weights[:, 0:1] * sar_feature
            +
            weights[:, 1:2] * optical_feature
        )

        return fused, weights


class FusionTransformer(nn.Module):
    def __init__(
        self,
        channels=512,
        token_count=256,
        heads=8,
        layers=2,
        dropout=0.1
    ):
        super().__init__()

        self.channels = channels
        self.token_count = token_count

        self.position = nn.Parameter(
            torch.zeros(
                1,
                token_count,
                channels
            )
        )

        layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=heads,
            dim_feedforward=channels * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=layers
        )

        self.norm = nn.LayerNorm(channels)

        nn.init.trunc_normal_(
            self.position,
            std=0.02
        )

    def forward(self, x):
        b, c, h, w = x.shape

        tokens = x.flatten(2).transpose(1, 2)

        if tokens.shape[1] != self.token_count:
            raise RuntimeError(
                f"Token count {tokens.shape[1]} "
                f"!= expected {self.token_count}"
            )

        tokens = tokens + self.position
        tokens = self.encoder(tokens)
        tokens = self.norm(tokens)

        return tokens.transpose(1, 2).reshape(
            b, c, h, w
        )


class AttentionGate(nn.Module):
    def __init__(self, gate_channels, skip_channels):
        super().__init__()

        inter_channels = max(skip_channels // 2, 16)

        self.gate = nn.Conv2d(
            gate_channels,
            inter_channels,
            kernel_size=1,
            bias=False
        )

        self.skip = nn.Conv2d(
            skip_channels,
            inter_channels,
            kernel_size=1,
            bias=False
        )

        self.psi = nn.Sequential(
            nn.GELU(),
            nn.Conv2d(
                inter_channels,
                1,
                kernel_size=1
            ),
            nn.Sigmoid()
        )

    def forward(self, gate_feature, skip_feature):
        attention = self.psi(
            self.gate(gate_feature)
            +
            self.skip(skip_feature)
        )

        return skip_feature * attention


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.attention = AttentionGate(
            in_channels,
            skip_channels
        )

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels + skip_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            make_group_norm(out_channels),
            nn.GELU(),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            make_group_norm(out_channels),
            nn.GELU(),

            SEBlock(out_channels)
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        skip = self.attention(
            x,
            skip
        )

        return self.conv(
            torch.cat(
                [x, skip],
                dim=1
            )
        )


class FloodShiftNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.sar_encoder = ModalityEncoder(2)
        self.optical_encoder = ModalityEncoder(10)

        self.routers = nn.ModuleList(
            [
                ReliabilityRouter(64),
                ReliabilityRouter(128),
                ReliabilityRouter(256),
                ReliabilityRouter(512),
            ]
        )

        self.transformer = FusionTransformer(
            channels=512,
            token_count=256,
            heads=8,
            layers=2,
            dropout=0.1
        )

        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)

        self.final_decoder = nn.Sequential(
            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            make_group_norm(64),
            nn.GELU(),
            SEBlock(64)
        )

        self.head = nn.Conv2d(
            64,
            num_classes,
            kernel_size=1
        )

    def forward(
        self,
        sar,
        optical,
        p_clear,
        return_router=False
    ):
        sar_features = self.sar_encoder(sar)
        optical_features = self.optical_encoder(optical)

        fused_features = []
        router_weights = []

        for sar_f, opt_f, router in zip(
            sar_features,
            optical_features,
            self.routers
        ):
            fused, weights = router(
                sar_f,
                opt_f,
                p_clear
            )

            fused_features.append(fused)
            router_weights.append(weights)

        x = self.transformer(
            fused_features[3]
        )

        x = self.decoder3(
            x,
            fused_features[2]
        )

        x = self.decoder2(
            x,
            fused_features[1]
        )

        x = self.decoder1(
            x,
            fused_features[0]
        )

        x = F.interpolate(
            x,
            scale_factor=2.0,
            mode="bilinear",
            align_corners=False
        )

        x = self.final_decoder(x)
        logits = self.head(x)

        if return_router:
            return {
                "logits": logits,
                "router_weights": router_weights
            }

        return logits


In [11]:
# ============================================================
# CELL: P4-C08 — MODEL SHAPE + PARAMETER CHECK
# ============================================================

check_model = FloodShiftNet(
    num_classes=NUM_CLASSES
)

with torch.no_grad():
    output = check_model(
        torch.zeros(2, 2, 256, 256),
        torch.zeros(2, 10, 256, 256),
        torch.full((2, 1, 256, 256), 0.8),
        return_router=True
    )

assert output["logits"].shape == (
    2, NUM_CLASSES, 256, 256
)

assert len(output["router_weights"]) == 4

for weights in output["router_weights"]:
    assert torch.allclose(
        weights.sum(dim=1),
        torch.ones_like(
            weights[:, 0]
        ),
        atol=1e-5
    )

parameter_count = sum(
    p.numel()
    for p in check_model.parameters()
    if p.requires_grad
)

print("=" * 104)
print("P4-C08 — ARCHITECTURE CHECK")
print("=" * 104)

print(
    "Logits:",
    tuple(output["logits"].shape)
)

print(
    "Router scales:",
    [
        tuple(w.shape)
        for w in output["router_weights"]
    ]
)

print(
    "Parameters:",
    f"{parameter_count:,}",
    f"({parameter_count / 1e6:.2f} M)"
)

print(
    "Fusion Transformer:",
    "512 channels, 16×16, 256 tokens"
)

del check_model
gc.collect()


/tmp/ipykernel_104/1210890595.py:239: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


P4-C08 — ARCHITECTURE CHECK
Logits: (2, 3, 256, 256)
Router scales: [(2, 2, 128, 128), (2, 2, 64, 64), (2, 2, 32, 32), (2, 2, 16, 16)]
Parameters: 30,343,974 (30.34 M)
Fusion Transformer: 512 channels, 16×16, 256 tokens


1282

## Differential learning rates

The proposed training strategy deliberately uses different learning rates:

| Parameter group | LR |
|---|---:|
| SAR encoder | `1e-4` |
| Optical encoder | `1e-4` |
| Reliability routers | `2e-4` |
| Fusion Transformer | `2e-4` |
| Attention decoder | `3e-4` |
| Segmentation head | `5e-4` |

The main architecture run keeps the **same CE+Dice loss family as the standard baselines** so the architecture comparison is not confounded by a different loss. Loss-function experiments belong in a separate controlled ablation.

In [12]:
# ============================================================
# CELL: P4-C09 — MAIN LOSS + METRICS
# ============================================================

CE_WEIGHT = 0.5
DICE_WEIGHT = 0.5

cross_entropy = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    ignore_index=IGNORE_INDEX
)


def multiclass_dice_loss(logits, target, eps=1e-6):
    probabilities = torch.softmax(
        logits,
        dim=1
    )

    valid = (
        target != IGNORE_INDEX
    )

    safe_target = target.clone()
    safe_target[~valid] = 0

    one_hot = F.one_hot(
        safe_target,
        num_classes=NUM_CLASSES
    ).permute(0, 3, 1, 2).float()

    valid_float = valid.unsqueeze(1).float()

    probabilities = probabilities * valid_float
    one_hot = one_hot * valid_float

    intersection = (
        probabilities * one_hot
    ).sum(dim=(0, 2, 3))

    denominator = (
        probabilities.sum(dim=(0, 2, 3))
        +
        one_hot.sum(dim=(0, 2, 3))
    )

    dice = (
        2.0 * intersection + eps
    ) / (
        denominator + eps
    )

    return 1.0 - dice.mean()


def segmentation_loss(logits, target):
    ce = cross_entropy(logits, target)
    dice = multiclass_dice_loss(logits, target)

    total = (
        CE_WEIGHT * ce
        +
        DICE_WEIGHT * dice
    )

    return total, ce.detach(), dice.detach()


def confusion_from_logits(logits, target):
    prediction = torch.argmax(
        logits,
        dim=1
    )

    valid = (
        target != IGNORE_INDEX
    )

    pred_v = prediction[valid].detach().cpu()
    target_v = target[valid].detach().cpu()

    encoded = (
        target_v * NUM_CLASSES
        +
        pred_v
    )

    cm = torch.bincount(
        encoded,
        minlength=NUM_CLASSES * NUM_CLASSES
    ).reshape(
        NUM_CLASSES,
        NUM_CLASSES
    )

    return cm.numpy().astype(np.int64)


def metrics_from_confusion(cm):
    cm = cm.astype(np.float64)

    tp = np.diag(cm)
    gt = cm.sum(axis=1)
    pred = cm.sum(axis=0)

    fp = pred - tp
    fn = gt - tp

    iou = tp / np.maximum(
        tp + fp + fn,
        1.0
    )

    precision = tp / np.maximum(
        tp + fp,
        1.0
    )

    recall = tp / np.maximum(
        tp + fn,
        1.0
    )

    dice = (
        2.0 * tp
        /
        np.maximum(
            2.0 * tp + fp + fn,
            1.0
        )
    )

    accuracy = (
        tp.sum()
        /
        np.maximum(
            cm.sum(),
            1.0
        )
    )

    result = {
        "mIoU": float(iou.mean()),
        "macro_dice": float(dice.mean()),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "accuracy": float(accuracy),
    }

    for c, name in enumerate(CLASS_NAMES):
        result[f"iou_{name}"] = float(iou[c])
        result[f"dice_{name}"] = float(dice[c])
        result[f"precision_{name}"] = float(precision[c])
        result[f"recall_{name}"] = float(recall[c])

    return result


In [14]:
# ============================================================
# CELL: P4-C10 — DIFFERENTIAL-LR OPTIMIZER + EPOCH FUNCTIONS
# ============================================================

MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6

WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0

LR_SAR_ENCODER = 1e-4
LR_OPT_ENCODER = 1e-4
LR_ROUTER = 2e-4
LR_TRANSFORMER = 2e-4
LR_DECODER = 3e-4
LR_HEAD = 5e-4


def unwrap_model(model):
    return (
        model.module
        if isinstance(model, nn.DataParallel)
        else model
    )


def make_optimizer(model):
    core = unwrap_model(model)

    groups = [
        {
            "params": core.sar_encoder.parameters(),
            "lr": LR_SAR_ENCODER,
            "name": "sar_encoder"
        },
        {
            "params": core.optical_encoder.parameters(),
            "lr": LR_OPT_ENCODER,
            "name": "optical_encoder"
        },
        {
            "params": core.routers.parameters(),
            "lr": LR_ROUTER,
            "name": "reliability_routers"
        },
        {
            "params": core.transformer.parameters(),
            "lr": LR_TRANSFORMER,
            "name": "fusion_transformer"
        },
        {
            "params":
                list(core.decoder3.parameters())
                + list(core.decoder2.parameters())
                + list(core.decoder1.parameters())
                + list(core.final_decoder.parameters()),
            "lr": LR_DECODER,
            "name": "attention_decoder"
        },
        {
            "params": core.head.parameters(),
            "lr": LR_HEAD,
            "name": "segmentation_head"
        },
    ]

    return torch.optim.AdamW(
        groups,
        weight_decay=WEIGHT_DECAY
    )


def run_epoch(model, loader, optimizer=None, scaler=None):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    running_loss = 0.0
    running_ce = 0.0
    running_dice = 0.0
    batches = 0

    confusion = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64
    )

    router_sar_sum = np.zeros(4, dtype=np.float64)
    router_opt_sum = np.zeros(4, dtype=np.float64)
    router_count = np.zeros(4, dtype=np.float64)

    for batch in loader:
        sar = batch["sar"].to(
            DEVICE,
            non_blocking=True
        )

        optical = batch["optical"].to(
            DEVICE,
            non_blocking=True
        )

        reliability = batch["reliability"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        if is_train:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):
                output = model(
                    sar,
                    optical,
                    reliability,
                    return_router=True
                )

                logits = output["logits"]

                loss, ce, dice_loss = segmentation_loss(
                    logits,
                    target
                )

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP_NORM
                )

                scaler.step(optimizer)
                scaler.update()

        running_loss += float(loss.detach().cpu())
        running_ce += float(ce.cpu())
        running_dice += float(dice_loss.cpu())
        batches += 1

        confusion += confusion_from_logits(
            logits,
            target
        )

        with torch.no_grad():
            for scale, weights in enumerate(
                output["router_weights"]
            ):
                router_sar_sum[scale] += float(
                    weights[:, 0].sum().detach().cpu()
                )

                router_opt_sum[scale] += float(
                    weights[:, 1].sum().detach().cpu()
                )

                router_count[scale] += float(
                    weights[:, 0].numel()
                )

    metrics = metrics_from_confusion(
        confusion
    )

    metrics.update({
        "loss": running_loss / max(batches, 1),
        "ce_loss": running_ce / max(batches, 1),
        "dice_loss": running_dice / max(batches, 1),
    })

    for scale in range(4):
        denom = max(
            router_count[scale],
            1.0
        )

        metrics[f"router_sar_s{scale+1}"] = (
            router_sar_sum[scale] / denom
        )

        metrics[f"router_opt_s{scale+1}"] = (
            router_opt_sum[scale] / denom
        )

    return metrics


@torch.no_grad()
def evaluate_model(model, loader):
    return run_epoch(
        model=model,
        loader=loader,
        optimizer=None,
        scaler=None
    )


In [15]:
# ============================================================
# CELL: P4-C11 — TRAIN MAIN FLOODSHIFT-RAW MODEL
# SINGLE-GPU SAFE VERSION — NO nn.DataParallel
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

train_loader, val_loader, test_loader = make_loaders(seed=SEED)

model = FloodShiftNet(num_classes=NUM_CLASSES).to(DEVICE)
assert not isinstance(model, nn.DataParallel)

optimizer = make_optimizer(model)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6
)

scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

CHECKPOINT_PATH = CHECKPOINT_ROOT / "FloodShift_Raw_Main_best.pt"
CHECKPOINT_EASY_PATH = Path("/kaggle/working/FloodShift_Raw_Main_best.pt")
HISTORY_PATH = HISTORY_ROOT / "FloodShift_Raw_Main_history.csv"

best_val_miou = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
train_start = time.perf_counter()

print("=" * 116)
print("P4-C11 — TRAIN FLOODSHIFT-RAW [SINGLE GPU]")
print("=" * 116)
print("Device:", DEVICE)
print("DataParallel: False")

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()

    train_metrics = run_epoch(model, train_loader, optimizer=optimizer, scaler=scaler)
    val_metrics = evaluate_model(model, val_loader)
    scheduler.step(val_metrics["mIoU"])

    learning_rates = {
        group.get("name", f"group_{i}"): group["lr"]
        for i, group in enumerate(optimizer.param_groups)
    }

    record = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"lr_{k}": v for k, v in learning_rates.items()},
        "epoch_sec": time.perf_counter() - epoch_start,
    }
    history.append(record)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"val mIoU {val_metrics['mIoU']:.4f} | "
        f"temp IoU {val_metrics['iou_temporary_flood']:.4f} | "
        f"perm IoU {val_metrics['iou_permanent_water']:.4f} | "
        f"router opt S4 {val_metrics['router_opt_s4']:.3f} | "
        f"{record['epoch_sec']:.1f}s"
    )

    if val_metrics["mIoU"] > best_val_miou + 1e-6:
        best_val_miou = val_metrics["mIoU"]
        best_epoch = epoch
        epochs_without_improvement = 0

        payload = {
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "val_metrics": val_metrics,
            "config": {
                "experiment": "FloodShift_Raw_Main",
                "hardware": "single_visible_GPU",
                "data_parallel": False,
                "sar_variant": "Raw SAR",
                "optical_bands": OPTICAL_BANDS,
                "reliability": "renormalized P(clear)",
                "loss": "0.5 weighted CE + 0.5 Dice",
                "differential_lr": learning_rates,
                "tile_size": TILE_SIZE,
                "batch_size": BATCH_SIZE,
            },
        }

        torch.save(payload, CHECKPOINT_PATH)
        torch.save(payload, CHECKPOINT_EASY_PATH)
        print("  ↳ Best checkpoint updated")
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}; best epoch = {best_epoch}")
        break

training_minutes = (time.perf_counter() - train_start) / 60.0

assert CHECKPOINT_PATH.exists()
assert CHECKPOINT_EASY_PATH.exists()

print("\nTraining complete ✅")
print("Best epoch          :", best_epoch)
print("Best validation mIoU:", best_val_miou)
print("Training minutes    :", training_minutes)
print("Checkpoint          :", CHECKPOINT_PATH)
print("Easy checkpoint     :", CHECKPOINT_EASY_PATH)
print("Checkpoint size MB  :", f"{CHECKPOINT_PATH.stat().st_size/(1024**2):.2f}")


/tmp/ipykernel_104/1210890595.py:239: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


P4-C11 — TRAIN FLOODSHIFT-RAW [SINGLE GPU]
Device: cuda:0
DataParallel: False
Epoch 01/30 | train loss 0.6642 | val mIoU 0.4400 | temp IoU 0.1925 | perm IoU 0.2534 | router opt S4 0.296 | 112.0s
  ↳ Best checkpoint updated
Epoch 02/30 | train loss 0.4882 | val mIoU 0.3695 | temp IoU 0.1733 | perm IoU 0.0920 | router opt S4 0.203 | 57.1s
Epoch 03/30 | train loss 0.4348 | val mIoU 0.4236 | temp IoU 0.1972 | perm IoU 0.1968 | router opt S4 0.184 | 55.8s
Epoch 04/30 | train loss 0.4189 | val mIoU 0.4614 | temp IoU 0.1675 | perm IoU 0.3884 | router opt S4 0.182 | 56.3s
  ↳ Best checkpoint updated
Epoch 05/30 | train loss 0.3853 | val mIoU 0.4276 | temp IoU 0.2894 | perm IoU 0.0405 | router opt S4 0.174 | 54.8s
Epoch 06/30 | train loss 0.3755 | val mIoU 0.4952 | temp IoU 0.2282 | perm IoU 0.3592 | router opt S4 0.198 | 54.6s
  ↳ Best checkpoint updated
Epoch 07/30 | train loss 0.3657 | val mIoU 0.5132 | temp IoU 0.3214 | perm IoU 0.2630 | router opt S4 0.186 | 56.8s
  ↳ Best checkpoint updat

In [16]:
# ============================================================
# CELL: P4-C12 — BEST CHECKPOINT TEST + ROUTER SUMMARY
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

unwrap_model(model).load_state_dict(
    checkpoint["state_dict"]
)

val_metrics = evaluate_model(
    model,
    val_loader
)

test_metrics = evaluate_model(
    model,
    test_loader
)

parameter_count = sum(
    p.numel()
    for p in unwrap_model(model).parameters()
    if p.requires_grad
)

result_row = {
    "model": "FloodShift-Raw",
    "parameters": parameter_count,
    "best_epoch": int(checkpoint["epoch"]),
    "training_minutes": float(training_minutes),
    **{
        f"val_{k}": v
        for k, v in val_metrics.items()
    },
    **{
        f"test_{k}": v
        for k, v in test_metrics.items()
    },
}

PROPOSED_RESULT_PATH = (
    METRIC_ROOT
    /
    "FloodShift_Raw_Main_result.csv"
)

pd.DataFrame(
    [result_row]
).to_csv(
    PROPOSED_RESULT_PATH,
    index=False
)

print("=" * 112)
print("FLOODSHIFT-RAW MAIN RESULT")
print("=" * 112)

print(
    "Validation mIoU       :",
    f"{val_metrics['mIoU']:.4f}"
)

print(
    "Test mIoU             :",
    f"{test_metrics['mIoU']:.4f}"
)

print(
    "Test non-flood IoU    :",
    f"{test_metrics['iou_non_flood']:.4f}"
)

print(
    "Test permanent IoU    :",
    f"{test_metrics['iou_permanent_water']:.4f}"
)

print(
    "Test temporary IoU    :",
    f"{test_metrics['iou_temporary_flood']:.4f}"
)

print("\nMean TEST optical routing weights:")

for scale in range(1, 5):
    print(
        f"  Scale {scale}: "
        f"{test_metrics[f'router_opt_s{scale}']:.4f}"
    )

print("\nSaved:", PROPOSED_RESULT_PATH)


FLOODSHIFT-RAW MAIN RESULT
Validation mIoU       : 0.5132
Test mIoU             : 0.6636
Test non-flood IoU    : 0.9524
Test permanent IoU    : 0.5569
Test temporary IoU    : 0.4813

Mean TEST optical routing weights:
  Scale 1: 0.4797
  Scale 2: 0.3586
  Scale 3: 0.2819
  Scale 4: 0.2613

Saved: /kaggle/working/FloodShift_Proposed/metrics/FloodShift_Raw_Main_result.csv


In [17]:
# ============================================================
# CELL: P4-C12A — CHECKPOINT DOWNLOAD / LOCAL BACKUP
# Run this immediately after C12, BEFORE calibration.
# ============================================================

from pathlib import Path
from IPython.display import FileLink, display
import shutil
import hashlib

source_checkpoint = Path(
    "/kaggle/working/FloodShift_Proposed/checkpoints/FloodShift_Raw_Main_best.pt"
)

easy_checkpoint = Path(
    "/kaggle/working/FloodShift_Raw_Main_best.pt"
)

assert source_checkpoint.exists(), "Best checkpoint missing."

if not easy_checkpoint.exists():
    shutil.copy2(source_checkpoint, easy_checkpoint)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_sha256 = sha256_file(easy_checkpoint)

print("=" * 108)
print("P4-C12A — TRAINED CHECKPOINT PROTECTED ✅")
print("=" * 108)
print("Path  :", easy_checkpoint)
print("Size  :", f"{easy_checkpoint.stat().st_size/(1024**2):.2f} MB")
print("SHA256:", checkpoint_sha256)
print("\nClick this link NOW and save the file locally:\n")

display(
    FileLink(
        str(easy_checkpoint),
        result_html_prefix="⬇️ Download trained checkpoint: "
    )
)

print("\nDo not restart/stop the session until this file is downloaded.")


P4-C12A — TRAINED CHECKPOINT PROTECTED ✅
Path  : /kaggle/working/FloodShift_Raw_Main_best.pt
Size  : 115.87 MB
SHA256: 462a799f9cff43c53364629a2fa09f894aa38a182e44a86bd3505067aa3b20bd

Click this link NOW and save the file locally:



/kaggle/working/FloodShift_Raw_Main_best.pt


Do not restart/stop the session until this file is downloaded.


## Calibration, uncertainty and abstention

After the best checkpoint is selected by validation mIoU, a **single scalar temperature** is fitted on validation pixels only. The test split is never used to fit calibration.

The calibrated probabilities are then used for:

- Expected Calibration Error (ECE)
- multiclass Brier score
- normalized predictive entropy
- confidence-based abstention / selective accuracy

Temperature scaling does **not** change the argmax segmentation map.

In [18]:
# ============================================================
# CELL: P4-C13 — VALIDATION TEMPERATURE SCALING [SINGLE GPU]
# ============================================================

assert not isinstance(model, torch.nn.DataParallel)
model.eval()

@torch.inference_mode()
def collect_calibration_samples(model, loader, max_per_batch=2048):
    logits_list = []
    labels_list = []
    rng = np.random.default_rng(SEED)

    for batch_index, batch in enumerate(loader, start=1):
        sar = batch["sar"].contiguous().to(DEVICE, non_blocking=True)
        optical = batch["optical"].contiguous().to(DEVICE, non_blocking=True)
        reliability = batch["reliability"].contiguous().to(DEVICE, non_blocking=True)
        target = batch["target"].contiguous().to(DEVICE, non_blocking=True)

        logits = model(sar, optical, reliability)
        valid = target != IGNORE_INDEX

        logits_valid = logits.permute(0, 2, 3, 1)[valid]
        labels_valid = target[valid]

        count = int(logits_valid.shape[0])
        if count > max_per_batch:
            chosen = torch.as_tensor(
                rng.choice(count, size=max_per_batch, replace=False),
                dtype=torch.long,
                device=DEVICE,
            )
            logits_valid = logits_valid[chosen]
            labels_valid = labels_valid[chosen]

        logits_list.append(logits_valid.float().cpu())
        labels_list.append(labels_valid.long().cpu())

        if batch_index == 1 or batch_index % 10 == 0 or batch_index == len(loader):
            print(f"[{batch_index:02d}/{len(loader)}] validation batches collected")

    return torch.cat(logits_list, dim=0), torch.cat(labels_list, dim=0)

calibration_logits_cpu, calibration_labels_cpu = collect_calibration_samples(
    model, val_loader, max_per_batch=2048
)

print("Calibration samples:", calibration_labels_cpu.numel())
print("Logit matrix       :", tuple(calibration_logits_cpu.shape))
assert torch.isfinite(calibration_logits_cpu).all()

calibration_logits = calibration_logits_cpu.to(DEVICE)
calibration_labels = calibration_labels_cpu.to(DEVICE)

log_temperature = nn.Parameter(torch.zeros((), device=DEVICE))

temperature_optimizer = torch.optim.LBFGS(
    [log_temperature], lr=0.05, max_iter=50, line_search_fn="strong_wolfe"
)

def calibration_closure():
    temperature_optimizer.zero_grad()
    temperature = torch.exp(log_temperature)
    loss = F.cross_entropy(calibration_logits / temperature, calibration_labels)
    loss.backward()
    return loss

initial_nll = float(F.cross_entropy(calibration_logits, calibration_labels).detach().cpu())
temperature_optimizer.step(calibration_closure)
TEMPERATURE = float(torch.exp(log_temperature).detach().cpu())
final_nll = float(F.cross_entropy(calibration_logits / TEMPERATURE, calibration_labels).detach().cpu())

assert np.isfinite(TEMPERATURE) and TEMPERATURE > 0.0

CALIBRATION_PATH = CALIBRATION_ROOT / "temperature_scaling.json"
with open(CALIBRATION_PATH, "w") as f:
    json.dump(
        {
            "temperature": TEMPERATURE,
            "fit_split": "validation",
            "test_used_for_fit": False,
            "sample_count": int(calibration_labels_cpu.numel()),
            "initial_validation_nll": initial_nll,
            "calibrated_validation_nll": final_nll,
            "single_gpu": True,
            "data_parallel": False,
        },
        f,
        indent=2,
    )

print("Initial validation NLL:", initial_nll)
print("Final validation NLL  :", final_nll)
print("Temperature           :", TEMPERATURE)
print("Saved                 :", CALIBRATION_PATH)
print("\nP4-C13 completed successfully ✅")


[01/34] validation batches collected
[10/34] validation batches collected
[20/34] validation batches collected
[30/34] validation batches collected
[34/34] validation batches collected
Calibration samples: 69632
Logit matrix       : (69632, 3)
Initial validation NLL: 0.21373574435710907
Final validation NLL  : 0.19804348051548004
Temperature           : 1.3621317148208618
Saved                 : /kaggle/working/FloodShift_Proposed/calibration/temperature_scaling.json

P4-C13 completed successfully ✅


In [20]:
# ============================================================
# CELL: P4-C14 — CALIBRATED TEST UNCERTAINTY + ABSTENTION
# ============================================================

ECE_BINS = 15
edges = np.linspace(
    0.0,
    1.0,
    ECE_BINS + 1
)

bin_count = np.zeros(ECE_BINS, dtype=np.float64)
bin_conf_sum = np.zeros(ECE_BINS, dtype=np.float64)
bin_correct_sum = np.zeros(ECE_BINS, dtype=np.float64)

brier_sum = 0.0
entropy_sum = 0.0
valid_pixel_count = 0

ABSTENTION_THRESHOLDS = np.array(
    [0.00, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95],
    dtype=np.float64
)

selected_count = np.zeros(
    len(ABSTENTION_THRESHOLDS),
    dtype=np.float64
)

selected_correct = np.zeros(
    len(ABSTENTION_THRESHOLDS),
    dtype=np.float64
)

model.eval()

with torch.no_grad():
    for batch in test_loader:
        sar = batch["sar"].to(
            DEVICE,
            non_blocking=True
        )

        optical = batch["optical"].to(
            DEVICE,
            non_blocking=True
        )

        reliability = batch["reliability"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            sar,
            optical,
            reliability
        )

        probabilities = torch.softmax(
            logits / TEMPERATURE,
            dim=1
        )

        confidence, prediction = probabilities.max(
            dim=1
        )

        valid = (
            target != IGNORE_INDEX
        )

        probs_v = probabilities.permute(
            0, 2, 3, 1
        )[valid]

        confidence_v = confidence[valid]
        prediction_v = prediction[valid]
        target_v = target[valid]

        correct = (
            prediction_v == target_v
        ).float()

        one_hot = F.one_hot(
            target_v,
            num_classes=NUM_CLASSES
        ).float()

        brier_sum += float(
            (
                (probs_v - one_hot) ** 2
            ).sum(
                dim=1
            ).sum().cpu()
        )

        entropy = -(
            probs_v
            *
            torch.log(
                torch.clamp(
                    probs_v,
                    min=1e-8
                )
            )
        ).sum(
            dim=1
        ) / math.log(NUM_CLASSES)

        entropy_sum += float(
            entropy.sum().cpu()
        )

        conf_np = confidence_v.detach().cpu().numpy()
        correct_np = correct.detach().cpu().numpy()

        valid_pixel_count += int(
            conf_np.size
        )

        bin_ids = np.clip(
            np.digitize(
                conf_np,
                edges
            ) - 1,
            0,
            ECE_BINS - 1
        )

        for b in range(ECE_BINS):
            mask = (
                bin_ids == b
            )

            if mask.any():
                bin_count[b] += mask.sum()
                bin_conf_sum[b] += conf_np[mask].sum()
                bin_correct_sum[b] += correct_np[mask].sum()

        for i, threshold in enumerate(
            ABSTENTION_THRESHOLDS
        ):
            selected = (
                conf_np >= threshold
            )

            selected_count[i] += selected.sum()
            selected_correct[i] += correct_np[selected].sum()


ece = 0.0

for b in range(ECE_BINS):
    if bin_count[b] > 0:
        mean_conf = (
            bin_conf_sum[b]
            /
            bin_count[b]
        )

        mean_acc = (
            bin_correct_sum[b]
            /
            bin_count[b]
        )

        ece += (
            bin_count[b]
            /
            valid_pixel_count
        ) * abs(
            mean_acc - mean_conf
        )

BRIER = (
    brier_sum
    /
    valid_pixel_count
)

MEAN_NORMALIZED_ENTROPY = (
    entropy_sum
    /
    valid_pixel_count
)

abstention_rows = []

for i, threshold in enumerate(
    ABSTENTION_THRESHOLDS
):
    count = selected_count[i]

    coverage = (
        count
        /
        valid_pixel_count
    )

    accuracy = (
        selected_correct[i]
        /
        max(count, 1.0)
    )

    abstention_rows.append(
        {
            "confidence_threshold": threshold,
            "coverage": coverage,
            "selective_accuracy": accuracy,
            "selective_error": 1.0 - accuracy,
        }
    )

abstention_table = pd.DataFrame(
    abstention_rows
)

UNCERTAINTY_METRIC_PATH = (
    CALIBRATION_ROOT
    /
    "calibrated_uncertainty_metrics.json"
)

ABSTENTION_PATH = (
    CALIBRATION_ROOT
    /
    "abstention_curve.csv"
)

with open(
    UNCERTAINTY_METRIC_PATH,
    "w"
) as f:
    json.dump(
        {
            "temperature": TEMPERATURE,
            "ece": float(ece),
            "multiclass_brier": float(BRIER),
            "mean_normalized_entropy": float(
                MEAN_NORMALIZED_ENTROPY
            ),
            "valid_test_pixels": int(
                valid_pixel_count
            ),
        },
        f,
        indent=2
    )

abstention_table.to_csv(
    ABSTENTION_PATH,
    index=False
)

print("=" * 104)
print("P4-C14 — CALIBRATED UNCERTAINTY")
print("=" * 104)
print("ECE                    :", ece)
print("Brier                  :", BRIER)
print("Mean normalized entropy:", MEAN_NORMALIZED_ENTROPY)

print("\nAbstention curve:")
print(
    abstention_table.to_string(
        index=False
    )
)

print("\nSaved:", UNCERTAINTY_METRIC_PATH)
print("Saved:", ABSTENTION_PATH)


P4-C14 — CALIBRATED UNCERTAINTY
ECE                    : 0.01854868151221218
Brier                  : 0.08924457495694241
Mean normalized entropy: 0.20009100970794383

Abstention curve:
 confidence_threshold  coverage  selective_accuracy  selective_error
                 0.00  1.000000            0.939883         0.060117
                 0.50  0.991972            0.943852         0.056148
                 0.60  0.954630            0.959045         0.040955
                 0.70  0.905213            0.972368         0.027632
                 0.80  0.859234            0.984459         0.015541
                 0.90  0.792451            0.995293         0.004707
                 0.95  0.717736            0.998893         0.001107

Saved: /kaggle/working/FloodShift_Proposed/calibration/calibrated_uncertainty_metrics.json
Saved: /kaggle/working/FloodShift_Proposed/calibration/abstention_curve.csv


## Counterfactual sensor attribution

The final evaluation asks how much the segmentation changes when one modality is removed without retraining:

- **full**: SAR + optical + `P(clear)`
- **SAR-off**: normalized SAR replaced by zero, which is the neutral training-mean value
- **optical-off**: normalized optical replaced by zero and `P(clear)` forced to zero

The mIoU drop and the change in temporary-flood probability provide a direct sensor-dependence measure.

In [21]:
# ============================================================
# CELL: P4-C15 — COUNTERFACTUAL SENSOR ATTRIBUTION
# ============================================================

counterfactual_confusions = {
    condition: np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64
    )
    for condition in [
        "full",
        "sar_off",
        "optical_off"
    ]
}

temp_delta_sar_off = 0.0
temp_delta_optical_off = 0.0
cf_valid_pixels = 0

model.eval()

with torch.no_grad():
    for batch in test_loader:
        sar = batch["sar"].to(
            DEVICE,
            non_blocking=True
        )

        optical = batch["optical"].to(
            DEVICE,
            non_blocking=True
        )

        reliability = batch["reliability"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )

        logits_full = model(
            sar,
            optical,
            reliability
        )

        logits_sar_off = model(
            torch.zeros_like(sar),
            optical,
            reliability
        )

        logits_optical_off = model(
            sar,
            torch.zeros_like(optical),
            torch.zeros_like(reliability)
        )

        counterfactual_confusions["full"] += confusion_from_logits(
            logits_full,
            target
        )

        counterfactual_confusions["sar_off"] += confusion_from_logits(
            logits_sar_off,
            target
        )

        counterfactual_confusions["optical_off"] += confusion_from_logits(
            logits_optical_off,
            target
        )

        valid = (
            target != IGNORE_INDEX
        )

        p_full = torch.softmax(
            logits_full / TEMPERATURE,
            dim=1
        )[:, 2]

        p_sar_off = torch.softmax(
            logits_sar_off / TEMPERATURE,
            dim=1
        )[:, 2]

        p_opt_off = torch.softmax(
            logits_optical_off / TEMPERATURE,
            dim=1
        )[:, 2]

        temp_delta_sar_off += float(
            torch.abs(
                p_full[valid]
                -
                p_sar_off[valid]
            ).sum().cpu()
        )

        temp_delta_optical_off += float(
            torch.abs(
                p_full[valid]
                -
                p_opt_off[valid]
            ).sum().cpu()
        )

        cf_valid_pixels += int(
            valid.sum().cpu()
        )

rows = []

for condition, cm in counterfactual_confusions.items():
    metrics = metrics_from_confusion(cm)

    rows.append(
        {
            "condition": condition,
            **metrics
        }
    )

counterfactual_table = pd.DataFrame(rows)

full_miou = float(
    counterfactual_table.loc[
        counterfactual_table["condition"] == "full",
        "mIoU"
    ].iloc[0]
)

counterfactual_table["mIoU_drop_vs_full"] = (
    full_miou
    -
    counterfactual_table["mIoU"]
)

COUNTERFACTUAL_PATH = (
    ATTRIBUTION_ROOT
    /
    "counterfactual_sensor_ablation.csv"
)

COUNTERFACTUAL_SUMMARY_PATH = (
    ATTRIBUTION_ROOT
    /
    "counterfactual_sensor_attribution.json"
)

counterfactual_table.to_csv(
    COUNTERFACTUAL_PATH,
    index=False
)

with open(
    COUNTERFACTUAL_SUMMARY_PATH,
    "w"
) as f:
    json.dump(
        {
            "mean_abs_temp_probability_change_sar_off":
                temp_delta_sar_off / cf_valid_pixels,

            "mean_abs_temp_probability_change_optical_off":
                temp_delta_optical_off / cf_valid_pixels,

            "test_valid_pixels":
                int(cf_valid_pixels),
        },
        f,
        indent=2
    )

print("=" * 112)
print("P4-C15 — COUNTERFACTUAL SENSOR ATTRIBUTION")
print("=" * 112)

print(
    counterfactual_table[
        [
            "condition",
            "mIoU",
            "iou_non_flood",
            "iou_permanent_water",
            "iou_temporary_flood",
            "mIoU_drop_vs_full"
        ]
    ].to_string(
        index=False
    )
)

print(
    "\nMean |Δ temp-flood probability|, SAR off:",
    temp_delta_sar_off / cf_valid_pixels
)

print(
    "Mean |Δ temp-flood probability|, optical off:",
    temp_delta_optical_off / cf_valid_pixels
)

print("\nSaved:", COUNTERFACTUAL_PATH)
print("Saved:", COUNTERFACTUAL_SUMMARY_PATH)


P4-C15 — COUNTERFACTUAL SENSOR ATTRIBUTION
  condition     mIoU  iou_non_flood  iou_permanent_water  iou_temporary_flood  mIoU_drop_vs_full
       full 0.663521       0.952405             0.556867             0.481290           0.000000
    sar_off 0.428422       0.926759             0.001184             0.357324           0.235098
optical_off 0.543798       0.897651             0.557643             0.176100           0.119722

Mean |Δ temp-flood probability|, SAR off: 0.048893906795356074
Mean |Δ temp-flood probability|, optical off: 0.10111470481053039

Saved: /kaggle/working/FloodShift_Proposed/attribution/counterfactual_sensor_ablation.csv
Saved: /kaggle/working/FloodShift_Proposed/attribution/counterfactual_sensor_attribution.json


In [23]:
# ============================================================
# CELL: P4-C16 — OPTIONAL MERGE WITH 04A BASELINE OUTPUT
# ============================================================

baseline_matches = list(
    INPUT_ROOT.rglob(
        "baseline_8model_final_comparison.csv"
    )
)

if baseline_matches:
    baseline_path = sorted(
        baseline_matches,
        key=lambda p: (
            len(str(p)),
            str(p)
        )
    )[0]

    baseline_table = pd.read_csv(
        baseline_path
    )

    proposed_table = pd.DataFrame(
        [
            {
                "model": "FloodShift-Raw",
                "parameters": parameter_count,
                "test_mIoU": test_metrics["mIoU"],
                "test_iou_non_flood":
                    test_metrics["iou_non_flood"],
                "test_iou_permanent_water":
                    test_metrics["iou_permanent_water"],
                "test_iou_temporary_flood":
                    test_metrics["iou_temporary_flood"],
                "test_macro_dice":
                    test_metrics["macro_dice"],
                "test_macro_precision":
                    test_metrics["macro_precision"],
                "test_macro_recall":
                    test_metrics["macro_recall"],
                "test_accuracy":
                    test_metrics["accuracy"],
            }
        ]
    )

    comparison = pd.concat(
        [
            baseline_table,
            proposed_table
        ],
        ignore_index=True,
        sort=False
    )

    comparison = comparison.sort_values(
        "test_mIoU",
        ascending=False
    ).reset_index(drop=True)

    COMPARISON_PATH = (
        METRIC_ROOT
        /
        "FloodShift_vs_8_standard_baselines.csv"
    )

    comparison.to_csv(
        COMPARISON_PATH,
        index=False
    )

    print(
        comparison[
            [
                "model",
                "test_mIoU",
                "test_iou_permanent_water",
                "test_iou_temporary_flood",
                "test_macro_dice"
            ]
        ].to_string(
            index=False
        )
    )

    print("\nSaved:", COMPARISON_PATH)

else:
    print(
        "04A baseline output is not attached — skipping comparison."
    )

    print(
        "This does NOT affect proposed-model training or evaluation."
    )


              model  test_mIoU  test_iou_permanent_water  test_iou_temporary_flood  test_macro_dice
     FloodShift-Raw   0.663555                  0.556947                  0.481309         0.780301
       SegFormer-B2   0.652444                  0.573947                  0.449395         0.771764
       SegFormer-B0   0.628324                  0.557339                  0.394592         0.749003
DeepLabV3+-ResNet50   0.627144                  0.565619                  0.386014         0.747729
       FPN-ResNet34   0.627096                  0.564438                  0.387602         0.747859
    PSPNet-ResNet50   0.617265                  0.559782                  0.375303         0.740030
       FCN-ResNet50   0.565675                  0.483549                  0.288494         0.686904
     U-Net-ResNet34   0.553726                  0.467473                  0.278591         0.676189
   U-Net++-ResNet34   0.553488                  0.474494                  0.278791         0.676985


In [24]:
# ============================================================
# CELL: P4-C17 — SAVE PROPOSED-MODEL PROTOCOL
# ============================================================

protocol = {
    "model": "FloodShift-Raw",
    "task": "3-class multimodal flood segmentation",

    "classes": {
        "0": "non_flood",
        "1": "permanent_water",
        "2": "temporary_flood",
        "255": "ignore"
    },

    "dataset": {
        "retained_scenes": 440,
        "train": 314,
        "validation": 68,
        "test": 58,
        "tile_size": TILE_SIZE,
        "tiles_per_scene": TILES_PER_SCENE,
    },

    "inputs": {
        "sar": ["VV", "VH"],
        "sar_variant": "Raw SAR",
        "optical": OPTICAL_BANDS,
        "optical_reliability": "renormalized P(clear)",
    },

    "architecture": {
        "separate_modality_encoders": True,
        "channel_attention": "SE",
        "multiscale_reliability_router": True,
        "router_weighting": "2-way spatial softmax",

        "fusion_transformer": {
            "channels": 512,
            "tokens": 256,
            "heads": 8,
            "layers": 2,
        },

        "attention_gated_skips": True,
        "num_classes": 3,
    },

    "training": {
        "optimizer": "AdamW",
        "weight_decay": WEIGHT_DECAY,
        "loss": "0.5 weighted CrossEntropy + 0.5 macro Dice",

        "differential_learning_rates": {
            "sar_encoder": LR_SAR_ENCODER,
            "optical_encoder": LR_OPT_ENCODER,
            "routers": LR_ROUTER,
            "transformer": LR_TRANSFORMER,
            "decoder": LR_DECODER,
            "head": LR_HEAD,
        },

        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "amp": AMP_ENABLED,
    },

    "uncertainty": {
        "temperature_scaling": True,
        "temperature_fit_split": "validation",
        "ece": True,
        "brier": True,
        "predictive_entropy": True,
        "abstention_curve": True,
    },

    "counterfactual_attribution": {
        "sar_off": True,
        "optical_off": True,
    },

    "preprocessing_policy":
        "Notebook 03 permanent global lock; no retuning"
}

PROTOCOL_PATH = (
    OUTPUT_ROOT
    /
    "FloodShift_proposed_model_protocol.json"
)

with open(
    PROTOCOL_PATH,
    "w"
) as f:
    json.dump(
        protocol,
        f,
        indent=2
    )

print("=" * 112)
print("FLOODSHIFT PROPOSED-MODEL NOTEBOOK COMPLETE ✅")
print("=" * 112)

print("Best checkpoint:", CHECKPOINT_PATH)
print("Main metrics   :", PROPOSED_RESULT_PATH)
print("Calibration    :", CALIBRATION_PATH)
print("Attribution    :", COUNTERFACTUAL_PATH)
print("Protocol       :", PROTOCOL_PATH)

print(
    "\nSave this Kaggle notebook version WITH outputs."
)


FLOODSHIFT PROPOSED-MODEL NOTEBOOK COMPLETE ✅
Best checkpoint: /kaggle/working/FloodShift_Proposed/checkpoints/FloodShift_Raw_Main_best.pt
Main metrics   : /kaggle/working/FloodShift_Proposed/metrics/FloodShift_Raw_Main_result.csv
Calibration    : /kaggle/working/FloodShift_Proposed/calibration/temperature_scaling.json
Attribution    : /kaggle/working/FloodShift_Proposed/attribution/counterfactual_sensor_ablation.csv
Protocol       : /kaggle/working/FloodShift_Proposed/FloodShift_proposed_model_protocol.json

Save this Kaggle notebook version WITH outputs.


# What comes after this notebook

Keep the main proposed-model run frozen, then perform the following as controlled studies:

1. SAR preprocessing ablation: Raw / Lee Sigma / Homomorphic BM3D / DP-SSD L0 / DP-SSD-Calibrated.
2. Architecture ablations: fixed fusion, no reliability prior, no Fusion Transformer, no channel attention, no skip attention.
3. Loss ablations: weighted CE, Dice, focal, Tversky and justified combinations.
4. Differential-LR ablation: same LR everywhere vs grouped LR.
5. Ensemble experiment only after individual configurations are frozen.
6. VLM / language stage consumes the frozen segmentation masks, calibrated uncertainty and sensor-attribution outputs.